Napredna analiza podataka - Projekat - Klasterizacija teksta

Detekcija Phising mejlova na osnovu sadržaja mejla

Uvod: Razraditi

Instalacija potrebnih biblioteka

In [26]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
import matplotlib.pyplot as plt
import spacy

Prvobitno ucitavanje dataset-a sa dodatnim informacijama o broju unikatnih vrednosti opservacija i NaN  vrednosti

In [28]:
data = pd.read_csv('./data/emails/Nazario.csv')
data

,sender,receiver,date,subject,body,urls,label
0,Mail System Internal Data <MAILER-DAEMON@monke...,NaN,28 Sep 2017 09:57:25 -0400,DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA,This text is part of the internal format of yo...,1,1
1,cPanel <service@cpanel.com>,jose@monkey.org,"Fri, 30 Oct 2015 00:00:48 -0500",Verify Your Account,Business with \t\t\t\t\t\t\t\tcPanel & WHM \t...,1,1
2,Microsoft Outlook <recepcao@unimedceara.com.br>,NaN,"Fri, 30 Oct 2015 06:21:59 -0300 (BRT)",Helpdesk Mailbox Alert!!!,Your two incoming mails were placed on pending...,1,1
3,Ann Garcia <AnGarcia@mcoe.org>,"""info@maaaaa.org"" <info@maaaaa.org>","Fri, 30 Oct 2015 14:54:33 +0000",IT-Service Help Desk,Password will expire in 3 days. Click Here To ...,0,1
4,"""USAA"" <usaaacctupdate@sccu4u.com>",Recipients <usaaacctupdate@sccu4u.com>,"Fri, 30 Oct 2015 14:02:33 -0500",Final USAA Reminder - Update Your Account Now,"To ensure delivery to your inbox, please add U...",1,1
...,...,...,...,...,...,...,...
1560,info advantageapartments.com <info@advantageap...,"""jose@monkey.org"" <jose@monkey.org>","Sun, 11 Dec 2022 12:04:50 +0000",Receipt for Your Payment to FTX.,PayPal You sent a payment of $699.99 USD to FT...,0,1
1561,monkey.org <HELPDESK@monkey.org>,jose@monkey.org,15 Dec 2022 18:33:00 -0500,Rectify Your Password With monkey.org,"monkey.org Hi jose,Pa⁠s⁠sword for⁠ jose@monke...",1,1
1562,Help Center\n\t<info.help-center.co.za_info.he...,jose@monkey.org,"Tue, 20 Dec 2022 23:00:27 +0000 (UTC)",Netflix : We're having some trouble with your ...,"HELLO, Please note that, your monthly paymen...",1,1
1563,MetaMask <info@sofamekar.com>,jose@monkey.org,27 Dec 2022 10:56:49 +0800,Your MetaMask wallet will be suspended,Verify your MetaMask Wallet Our system has sho...,1,1


Pre same analize, potrebno je da očistim dataset, prvo proveravam koliko je postojećih NaN vrednosti i da li mogu da ih odstranim

In [29]:
data.isna().sum()

sender       0
receiver    96
date         1
subject      4
body         0
urls         0
label        0
dtype: int64

Najveći problem predstavlja atribut primalac koji sadrži NaN vrednosti i prikrivene NaN vrednosti koje sam uočio prolazeći kroz data-set. Primer ovakve vrednosti je Undisclosed recipients:;

Proveriću koliko ima unikatnih vrednosti za atribut primaoca

In [30]:
data['receiver'].value_counts()

receiver
jose@monkey.org                                  938
undisclosed-recipients:;                          39
<jose@monkey.org>                                 36
"jose@monkey.org" <jose@monkey.org>               26
"jose" <jose@monkey.org>                          13
                                                ... 
Recipients <s.moran@whitgift.co.uk>                1
Recipients <usaa-suport@sa4u.net>                  1
Recipients <USAA.protectionsupport@uusaa.com>      1
Recipients <usa-security@whosay.com>               1
Recipients                                         1
Name: count, Length: 356, dtype: int64

Na osnovu prethodnog izveštaja vidi se da je dominantna adresa jose@monkey.org, čitajući na internetu sam saznao da adresa služila kao honeypot za prikupljanje phising mejlova. Sa jedne strane postoje mejlovi čiji primaoci nisu navedeni u maloj količini, dok sa druge strane postoji honeypot primaoc. Ovakva raspodela vrednosti atributa primaoca govori dominatnost pojavljivanja honeypot-a, tako da smatram da ovaj atribut sa ovom raspodelom vrednosti neće biti značajan u razlikovanju klasa

In [32]:
data.drop(['receiver'],axis=1,inplace=True)

In [33]:
data.sample(n=5)

,sender,date,subject,body,urls,label
759,Email Server Admin <secureserver@servers.com>,20 Apr 2018 05:38:45 -0700,Hi jose is holding (5) new unread messages o...,"monkey.org Hi jose, Our server security is ...",1,1
59,"""RIVERS, SHAREECE R"" <srrivers@email.sc.edu>","Mon, 21 Dec 2015 15:39:55 +0000",Warning ! SRX1299917120ID,Your e-mail account was LOGIN today by Unknown...,0,1
1412,"""Help Center"" <support@barcast.app>","Wed, 18 May 2022 16:42:59 +0000",We're having some trouble with your current bi...,"Hi jose@monkey.org, We're having some trouble...",1,1
830,Brain Lee <sales@tresnojatifurniture.com>,"Mon, 20 Aug 2018 10:50:01 -0400",Re: Request For Quotation,"WARM GREETINGS, I HAVE SENT SEVERAL EMAILS W...",0,1
758,"COMUNICADO@www.lee.com, IMPORTANTE@www.lee.com","Fri, 20 Apr 2018 01:37:46 +0300 (EEST)",santender@infoemail.com,UwdabmaselMwdabmaselA QwdabmaselUESwdabmaselTÃ...,1,1


Ostali atributi imaju jako mali broj NaN vrednosti u odnosu na ceo dataset(< 1%), tako da smatram da je najbolja opcija da obrišem ove opservacije

In [38]:
subjects_to_drop = data.loc[data['subject'].isna() == True]
dates_to_drop = data.loc[data['date'].isna() == True]
data.drop(subjects_to_drop.index,inplace=True)
data.drop(dates_to_drop.index,inplace=True)

In [39]:
data.isna().sum()

sender     0
date       0
subject    0
body       0
urls       0
label      0
dtype: int64

Datum ću zasad izostaviti iz analize, moguće je kreirati kategoriju na osnovu dana u nedelji, to takodje možda može doprineti razdvajanju klasa

Takodje treba proveriti atribut url i njegov doprinos razdvajanju klasa

In [43]:
data['urls'].value_counts()

urls
1    1419
0     141
Name: count, dtype: int64

Atribut url je binarnog tipa, 1 oznacava da mejl poseduje url u sebi, a 0 da ne poseduje. Postojanje Url-a može biti značajan indikator u detektovanju phishing-a, tako da ću ga detaljnije analizirati

In [47]:
data_sub = data[['sender','subject','body','urls','label']]
data_sub.loc[data_sub['label'] == 0]

,sender,subject,body,urls,label
